In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Prices per million tokens (USD). Source: claude.com/pricing
PRICING = {
    "claude-opus-4-7":   {"input": 5.00, "output": 25.00},
    "claude-opus-4-6":   {"input": 5.00, "output": 25.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5-20251001":  {"input": 1.00, "output": 5.00},
}

def cost_of_call(response, batch=False):
    rates = PRICING[response.model]
    u = response.usage

    cost = (u.input_tokens / 1_000_000) * rates["input"]
    cost += (u.output_tokens / 1_000_000) * rates["output"]

    # Cache reads are ~10% of base input price
    cache_read = getattr(u, "cache_read_input_tokens", 0) or 0
    cost += (cache_read / 1_000_000) * rates["input"] * 0.10

    # Cache writes (5-min) are ~25% above base input
    cache_write = getattr(u, "cache_creation_input_tokens", 0) or 0
    cost += (cache_write / 1_000_000) * rates["input"] * 1.25

    if batch:
        cost *= 0.5

    out = f"""
        Input tokens: {u.input_tokens}
        Output tokens: {u.output_tokens}
        Total tokens: {u.input_tokens + u.output_tokens}
        Cost: {cost}
    """
    return out


In [ ]:
# Create an API cleint

client = Anthropic()
model = "claude-haiku-4-5"


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What type of AI powered apps can I build using Claude API ? Answer in one line."
    }
]

In [ ]:
# Make a request

response = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages)

In [ ]:
print(response)